# CLOY Processing

This notebook is for me to work with the CLOY Reddit data. The goal is to turn each thread into a vector of words, compare the threads using cosine similarity, and find the most similar threads.

I also want to see whether removing stop words changes the similarity results.

## Table of Contents

1. [Importing Libraries](#importing-libraries)
2. [Reading the CLOY Data](#reading-the-cloy-data)
3. [Looking at One Thread](#looking-at-one-thread)
4. [Looking at the Comments](#looking-at-the-comments)
5. [Collecting Nested Comments](#collecting-nested-comments)
6. [Combining a Thread into Text](#combining-a-thread-into-text)
7. [Processing the Words](#processing-the-words)
8. [Processing All Threads](#processing-all-threads)
9. [Turning Threads into Word Vectors](#turning-threads-into-word-vectors)
10. [Calculating Cosine Similarity](#calculating-cosine-similarity)
11. [Saving the Similarity Data](#saving-the-similarity-data)
12. [Finding the Top 5 Similar Threads](#finding-the-top-5-similar-threads)
13. [Removing Stop Words](#removing-stop-words)
14. [Comparing the Results](#comparing-the-results)
15. [Reflection](#reflection)

## Importing Libraries

First, I imported the libraries I need for reading the JSONL file, working with the data, processing the text, and calculating similarity.

In [70]:
import json
import pandas as pd
import numpy as np
from numpy.linalg import norm

from textblob import TextBlob

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Reading the CLOY Data

Before trying to process everything, I wanted to first read the JSONL file and see what the data actually looks like.

In [71]:
file_path = "data/cs315-week4-data/RedditExplorer/cloy_chronological.jsonl"

Since this is a JSONL file, each line is its own JSON object. I read each line and added it to a list.

In [72]:
threads = []

with open(file_path, "r") as file:
    for line in file:
        thread = json.loads(line)
        threads.append(thread)

len(threads)

3075

There are 3,075 threads in the file.

## Looking at One Thread

Before writing any functions, I looked at one thread to understand how it is organized.

In [73]:
threads[0]
threads[0].keys()

dict_keys(['post_id', 'timestamp', 'author', 'subreddit', 'title', 'selftext', 'url', 'score', 'permalink', 'comments'])

I can see that each thread has information about the main post and also has a `comments` field. The fields I care about most for comparing the text are the title, the main post text, and the comments.

In [74]:
threads[0]["post_id"]
threads[0]["title"]
threads[0]["selftext"]
threads[0]["comments"]

[]

The first thread does not have any comments, so I decided to look at another thread that does.

## Looking at the Comments

The second thread has comments, so I used that one to understand how the comment data is stored.

In [75]:
threads[1]["title"]
len(threads[1]["comments"])
threads[1]["comments"][0]

{'comment_id': 'fk0rooq',
 'parent_id': 't3_ffh52z',
 'author': 'chimichangarolls',
 'comment_body': "i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.",
 'score': 19,
 'created_utc': 1583764623,
 'replies': [{'comment_id': 'fk1c27d',
   'parent_id': 't1_fk0rooq',
   'author': 'bruisedlee123',
   'comment_body': 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️',
   'score': 2,
   'created_utc': 1583776621,
   'replies': [{'comment_id': 'fsfm0lq',
     'parent_id': 't1_fk1c27d',
     'author': 'tmstms',
     'comment_body': "I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.",
     'score': 2,


A comment has its own information, including `comment_body`. I also noticed that some comments have a `replies` field, which means the structure is nested.

In [76]:
threads[1]["comments"][0].keys()
threads[1]["comments"][0]["comment_body"]
threads[1]["comments"][0]["replies"]

[{'comment_id': 'fk1c27d',
  'parent_id': 't1_fk0rooq',
  'author': 'bruisedlee123',
  'comment_body': 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️',
  'score': 2,
  'created_utc': 1583776621,
  'replies': [{'comment_id': 'fsfm0lq',
    'parent_id': 't1_fk1c27d',
    'author': 'tmstms',
    'comment_body': "I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.",
    'score': 2,
    'created_utc': 1590941768}]}]

This was important because I cannot just loop through the top-level comments. If I want all of the text in a thread, I also need to collect replies, including replies to replies.

## Collecting Nested Comments

Since replies can be nested inside other replies, I made a function that collects the comment text and then checks whether that comment has any replies.

The function calls itself again for the replies so it can keep going until there are no more replies.

In [77]:
def get_comment_text(comments):
    text = []
    
    for comment in comments:
        text.append(comment.get("comment_body", ""))
        
        if "replies" in comment:
            text.extend(get_comment_text(comment["replies"]))
    
    return text

Before using the function on every thread, I tested it on one thread to make sure it was actually collecting the comments.

In [78]:
test_comments = get_comment_text(threads[1]["comments"])
len(test_comments)
test_comments[:5]

["i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.",
 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️',
 "I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.",
 'I love this show. I stumbled across it on Netflix, gave it a try, and fell in love. I finished it this morning and already miss it. I love Yoon Se-Ri and Jeong Hyeok and nearly all of the other characters. I was so happy with the ending. I’ve added two more kdramas to my Netflix list.',
 'Which ones!']

The function gives me a list of the comment text, including nested replies.

## Combining a Thread into Text

Next, I wanted to turn one whole Reddit thread into one piece of text. I decided to include the title, the main post, and all of its comments and replies.

In [79]:
def get_thread_text(thread):
    title = thread.get("title", "")
    post = thread.get("selftext", "")
    comments = get_comment_text(thread.get("comments", []))
    
    all_text = [title, post] + comments
    
    return " ".join(all_text)

In [80]:
test_text = get_thread_text(threads[1])
test_text[:1000]

"ok, let’s get this started! What are your thoughts about the show? I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).\n\nCLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. \n\nI’m planning on doing a rewatch soon, when I get a chance! i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY a

Now the title, post, comments, and replies are all combined into one string. This makes it easier to process each thread as one document.

## Processing the Words

Next, I used TextBlob to split the thread into words. I also made the words lowercase so that words like "Love" and "love" are treated as the same word.

In [81]:
blob = TextBlob(test_text)
blob.words[:20]

WordList(['ok', 'let', '’', 's', 'get', 'this', 'started', 'What', 'are', 'your', 'thoughts', 'about', 'the', 'show', 'I', 'haven', '’', 't', 'seen', 'a'])

TextBlob already removes punctuation from `.words`, but I also only kept words that contain letters.

In [82]:
test_words = []

for word in blob.words:
    word = word.lower()
    
    if word.isalpha():
        test_words.append(word)

In [83]:
test_words[:20]

['ok',
 'let',
 's',
 'get',
 'this',
 'started',
 'what',
 'are',
 'your',
 'thoughts',
 'about',
 'the',
 'show',
 'i',
 'haven',
 't',
 'seen',
 'a',
 'kdrama',
 'for']

Now I have a cleaned list of lowercase words for the thread.

After testing those steps on one thread, I put the text processing into a function so I can do the same thing for every thread.

In [84]:
def process_thread(thread):
    text = get_thread_text(thread)
    blob = TextBlob(text)
    
    words = []
    
    for word in blob.words:
        word = word.lower()
        
        if word.isalpha():
            words.append(word)
    
    return words

In [85]:
process_thread(threads[1])[:20]

['ok',
 'let',
 's',
 'get',
 'this',
 'started',
 'what',
 'are',
 'your',
 'thoughts',
 'about',
 'the',
 'show',
 'i',
 'haven',
 't',
 'seen',
 'a',
 'kdrama',
 'for']

This gives me the cleaned words for one complete thread.

## Processing All Threads

Now that the function worked for one thread, I used it on all of the CLOY threads.

In [86]:
thread_data = []

for thread in threads:
    words = process_thread(thread)
    
    thread_data.append({
        "thread_id": thread["post_id"],
        "title": thread["title"],
        "words": words
    })

In [87]:
len(thread_data)

3075

In [88]:
thread_data[0]

{'thread_id': 'dkxewd',
 'title': 'Crash Landing On You has been created',
 'words': ['crash',
  'landing',
  'on',
  'you',
  'has',
  'been',
  'created',
  'deleted']}

For each thread, I kept the thread ID, title, and processed words. This gives me the information I need for both the similarity calculation and the final top 5 results.

In [89]:
threads_df = pd.DataFrame(thread_data)
threads_df.head()

,thread_id,title,words
0,dkxewd,Crash Landing On You has been created,"[crash, landing, on, you, has, been, created, ..."
1,ffh52z,"ok, let’s get this started! What are your thou...","[ok, let, s, get, this, started, what, are, yo..."
2,fguorr,Find someone who’ll make you noodles just like...,"[find, someone, who, ll, make, you, noodles, j..."
3,fhcrpl,Captain Ri’s squad! Who’s your favorite?,"[captain, ri, s, squad, who, s, your, favorite..."
4,fj5byj,Why I think we all need a second season of CLO...,"[why, i, think, we, all, need, a, second, seas..."


I put the processed data into a dataframe so it is easier to work with.

## Turning Threads into Word Vectors

The slides showed that the first step is to create a vocabulary of all of the unique words. Each word becomes one dimension in the vector.

Then each thread can be represented using the number of times each vocabulary word appears.

Before using this idea on all 3,075 threads, I wanted to make one small vector myself so I could see how the process from the slides works.

In [90]:
small_vocabulary = set()

for word in threads_df.iloc[0]["words"]:
    small_vocabulary.add(word)

small_vocabulary = sorted(list(small_vocabulary))

In [91]:
small_vocabulary[:20]

['been', 'crash', 'created', 'deleted', 'has', 'landing', 'on', 'you']

I used a set so that each word only appears once in the vocabulary.

In [92]:
test_vector = []

for word in small_vocabulary:
    count = threads_df.iloc[0]["words"].count(word)
    test_vector.append(count)

In [93]:
test_vector[:20]

[1, 1, 1, 1, 1, 1, 1, 1]

Each number in the vector shows how many times that vocabulary word appears in the thread. This is the term-frequency idea from the slides.

In [94]:
processed_text = []

for words in threads_df["words"]:
    processed_text.append(" ".join(words))

threads_df["processed_text"] = processed_text

In [95]:
threads_df[["thread_id", "title", "processed_text"]].head()

,thread_id,title,processed_text
0,dkxewd,Crash Landing On You has been created,crash landing on you has been created deleted
1,ffh52z,"ok, let’s get this started! What are your thou...",ok let s get this started what are your though...
2,fguorr,Find someone who’ll make you noodles just like...,find someone who ll make you noodles just like...
3,fhcrpl,Captain Ri’s squad! Who’s your favorite?,captain ri s squad who s your favorite i ca na...
4,fj5byj,Why I think we all need a second season of CLO...,why i think we all need a second season of clo...


Doing this manually for every word in every thread would take a long time, so I used CountVectorizer to do the same term-frequency process for the full dataset.

`CountVectorizer` creates one column for each word and counts how many times the word appears in each thread.

In [96]:
vectorizer = CountVectorizer()

In [97]:
word_vectors = vectorizer.fit_transform(threads_df["processed_text"])
word_vectors.shape

(3075, 26160)

The number of rows is the number of threads, while the number of columns is the number of different words found across the threads.

In [98]:
vectorizer.get_feature_names_out()[:20]

array(['aa', 'aaa', 'aaaa', 'aaaaa', 'aaaaaaa', 'aaaaaaaaaaaaa',
       'aaaaaaaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaaaaaaaaaa',
       'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaahhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh',
       'aaaaaaaaaaawwwwwwwwwweee',
       'aaaaaaaaaahhhhhhhhhhhyyyyaaaaaayyyyyyyyyy', 'aaaaaah', 'aaaaah',
       'aaaaahhhh', 'aaaaat', 'aaaah', 'aaaahhh', 'aaaahhhh', 'aaaand',
       'aaaargh'], dtype=object)

These are some of the words that became features in the vectors.

## Calculating Cosine Similarity

The slides showed that cosine similarity compares two vectors using their dot product and magnitudes. The smaller the angle between the vectors, the more similar they are.

Before calculating all of the similarities, I tested the calculation on two vectors.

In [99]:
def calculate_cosine_similarity(vector1, vector2):
    v1 = np.array(vector1)
    v2 = np.array(vector2)

    cosine = np.dot(v1, v2) / (norm(v1) * norm(v2))

    return cosine

In [100]:
calculate_cosine_similarity(test_vector, test_vector)

0.9999999999999998

A vector compared with itself has a cosine similarity of 1, which makes sense because they are identical.

For all 3,075 threads, I used the library version to calculate the full similarity matrix.

In [101]:
similarity_matrix = cosine_similarity(word_vectors)
similarity_matrix.shape

(3075, 3075)

The similarity matrix compares every thread with every other thread, so there is one row and one column for each of the 3,075 threads.

In [102]:
similarity_matrix[0][0]

0.9999999999999999

A thread has a similarity score of 1 with itself, which makes sense because the text is identical.

I used the thread IDs as the row and column labels so I can tell which threads the similarity scores belong to.

In [103]:
thread_ids = threads_df["thread_id"].tolist()

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=thread_ids,
    columns=thread_ids
)

similarity_df.head()


,dkxewd,ffh52z,fguorr,fhcrpl,fj5byj,fjimic,fjkurx,fkuhp7,flxz1f,fmbrra,...,1s0qkdq,1s10elb,1s10kr2,1s10qcn,1s13t5y,1s637j8,1s7yg7f,1s8aeiv,1s8i9nn,1s8ju38
dkxewd,1.000000,0.072625,0.031623,0.011158,0.068340,0.033710,0.019118,0.084655,0.101717,0.039778,...,0.069484,0.070318,0.133631,0.136062,0.134693,0.024332,0.068455,0.016188,0.092319,0.044901
ffh52z,0.072625,1.000000,0.517969,0.771452,0.735364,0.470049,0.762358,0.826678,0.561097,0.464883,...,0.813179,0.485540,0.059722,0.503884,0.647119,0.772236,0.805565,0.639556,0.318383,0.247831
fguorr,0.031623,0.517969,1.000000,0.485519,0.429916,0.349649,0.372411,0.487476,0.341875,0.181136,...,0.544577,0.448288,0.067612,0.428701,0.466827,0.461675,0.537364,0.409530,0.140130,0.159029
fhcrpl,0.011158,0.771452,0.485519,1.000000,0.633622,0.391183,0.637398,0.721126,0.508462,0.426090,...,0.728114,0.418041,0.059642,0.430613,0.548262,0.696132,0.737045,0.566448,0.307655,0.236477
fj5byj,0.068340,0.735364,0.429916,0.633622,1.000000,0.421432,0.590201,0.770500,0.546044,0.424799,...,0.793083,0.477222,0.063318,0.510346,0.561134,0.703731,0.776181,0.490901,0.417240,0.291309


## Saving the Similarity Data

The assignment asks me to save the similarity dataframe as a CSV file. I saved it inside the data folder so it does not get uploaded to GitHub.

In [104]:
similarity_df.to_csv("data/thread_similarity.csv")

Now I have a CSV containing the similarity score between every pair of threads.

## Finding the Top 5 Similar Threads

The next step is to find the five most similar threads for a given thread ID.

I first tested the idea using one thread ID before putting it into a function.

In [105]:
test_id = threads_df.iloc[1]["thread_id"]
test_id

similarity_df.loc[test_id].head()

dkxewd    0.072625
ffh52z    1.000000
fguorr    0.517969
fhcrpl    0.771452
fj5byj    0.735364
Name: ffh52z, dtype: float64

This gives all of the similarity scores for that thread, including its similarity with itself.

In [106]:
similarity_df.loc[test_id].sort_values(ascending=False).head(6)

ffh52z     1.000000
iy5qeu     0.920932
i42ez5     0.920669
17tpx8l    0.916004
lbo5y8     0.904402
1q62b7z    0.903829
Name: ffh52z, dtype: float64

I used six here because the first result is the original thread itself. I do not want to include that when I find the five most similar threads.

After testing the steps, I put them into a function. The function takes a thread ID and prints the five most similar thread IDs and titles.

In [107]:
def find_similar_threads(thread_id):
    scores = similarity_df.loc[thread_id]
    scores = scores.sort_values(ascending=False)
    
    top_five = scores.iloc[1:6]
    
    print("Original thread:")
    
    original_title = threads_df[
        threads_df["thread_id"] == thread_id
    ]["title"].iloc[0]
    
    print(thread_id, "-", original_title)
    print()
    print("Top 5 similar threads:")
    
    for similar_id in top_five.index:
        title = threads_df[
            threads_df["thread_id"] == similar_id
        ]["title"].iloc[0]
        
        print(similar_id, "-", title)

In [108]:
find_similar_threads(test_id)

Original thread:
ffh52z - ok, let’s get this started! What are your thoughts about the show?

Top 5 similar threads:
iy5qeu - Withdrawal, anyone?
i42ez5 - How come you started watching it?
17tpx8l - I just finished “Crash Landing On You” and I will never be the same.
lbo5y8 - HELP ME MOVE ON
1q62b7z - Surreal experience!


I also wanted to test the function on a thread with a more specific topic so it would be easier to judge whether the results actually seem related.

In [109]:
find_similar_threads("fj5byj")

Original thread:
fj5byj - Why I think we all need a second season of CLOY. (Spoilers)

Top 5 similar threads:
hsghrr - Second Season
g87z3m - Crash Landin on You: Potential season 2?
olj6u1 - If there was a second season , what would you like to see?
jg3333 - I think RJH and YSR get their happily ever after
jz55ap - Just finished CLOY, I really hope they don't do second season and ruin the happy ending


## Verifying the Results

I used the thread explorer to search for some of the thread IDs returned by the function and read the actual posts. I wanted to see whether the threads that had high similarity scores were actually talking about similar topics.

For the thread about wanting a second season, all five of the most similar threads were also about a possible second season of CLOY. Some were asking what people would want to see in another season, while others were discussing whether there should even be a second season. This made the results seem pretty accurate.

## Removing Stop Words

Since TextBlob is built on NLTK, I used NLTK's English stop word list to remove common words like "the," "and," "is," and "to." These words appear in a lot of threads but usually do not tell me much about what the thread is actually about.

I wanted to see whether removing them would make the similarity results more focused on the actual topics being discussed.

In [110]:
import nltk
nltk.download("stopwords")

/students/ao106/.local/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/students/ao106/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to
[nltk_data]     /students/ao106/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [111]:
from nltk.corpus import stopwords

In [112]:
stop_words = set(stopwords.words("english"))
list(stop_words)[:20]

['for',
 'while',
 'same',
 'very',
 "didn't",
 "wasn't",
 'isn',
 'who',
 'before',
 'between',
 'does',
 "doesn't",
 'such',
 'both',
 'having',
 'haven',
 'here',
 "it'd",
 "you'd",
 "should've"]

I made another version of my processing function. This one does the same cleaning as before but skips words that are in the stop word list.

In [113]:
def process_thread_no_stopwords(thread):
    text = get_thread_text(thread)
    blob = TextBlob(text)
    
    words = []
    
    for word in blob.words:
        word = word.lower()
        
        if word.isalpha() and word not in stop_words:
            words.append(word)
    
    return words

In [114]:
process_thread_no_stopwords(threads[1])[:20]

['ok',
 'let',
 'get',
 'started',
 'thoughts',
 'show',
 'seen',
 'kdrama',
 'longest',
 'time',
 'actually',
 'dad',
 'chose',
 'stream',
 'cloy',
 'netflix',
 'stopped',
 'watching',
 'second',
 'episode']

Compared with the original word list, common words like "the" and "and" are now removed.

Next, I processed all of the threads again without stop words.

In [115]:
words_no_stopwords = []

for thread in threads:
    words = process_thread_no_stopwords(thread)
    words_no_stopwords.append(words)

In [116]:
threads_df["words_no_stopwords"] = words_no_stopwords

In [117]:
processed_no_stopwords = []

for words in threads_df["words_no_stopwords"]:
    processed_no_stopwords.append(" ".join(words))

threads_df["processed_no_stopwords"] = processed_no_stopwords

In [118]:
threads_df[
    ["title", "processed_text", "processed_no_stopwords"]
].head()

,title,processed_text,processed_no_stopwords
0,Crash Landing On You has been created,crash landing on you has been created deleted,crash landing created deleted
1,"ok, let’s get this started! What are your thou...",ok let s get this started what are your though...,ok let get started thoughts show seen kdrama l...
2,Find someone who’ll make you noodles just like...,find someone who ll make you noodles just like...,find someone make noodles like ri jeong man no...
3,Captain Ri’s squad! Who’s your favorite?,captain ri s squad who s your favorite i ca na...,captain ri squad favorite ca narrow one love c...
4,Why I think we all need a second season of CLO...,why i think we all need a second season of clo...,think need second season cloy spoilers binged ...


I created a new set of word vectors using the text without stop words.

In [119]:
vectorizer_no_stopwords = CountVectorizer()

In [120]:
word_vectors_no_stopwords = vectorizer_no_stopwords.fit_transform(
    threads_df["processed_no_stopwords"]
)

In [121]:
word_vectors_no_stopwords.shape

(3075, 26019)

There are fewer word features now because the common stop words have been removed.

In [122]:
similarity_no_stopwords = cosine_similarity(word_vectors_no_stopwords)

In [123]:
similarity_no_stopwords_df = pd.DataFrame(
    similarity_no_stopwords,
    index=thread_ids,
    columns=thread_ids
)

In [124]:
similarity_no_stopwords_df.to_csv(
    "data/thread_similarity_no_stopwords.csv"
)

I made another version of the top 5 function that uses the similarity scores after removing stop words.

In [125]:
def find_similar_threads_no_stopwords(thread_id):
    scores = similarity_no_stopwords_df.loc[thread_id]
    scores = scores.sort_values(ascending=False)
    
    top_five = scores.iloc[1:6]
    
    print("Original thread:")
    
    original_title = threads_df[
        threads_df["thread_id"] == thread_id
    ]["title"].iloc[0]
    
    print(thread_id, "-", original_title)
    print()
    print("Top 5 similar threads:")
    
    for similar_id in top_five.index:
        title = threads_df[
            threads_df["thread_id"] == similar_id
        ]["title"].iloc[0]
        
        print(similar_id, "-", title)

In [126]:
find_similar_threads_no_stopwords("fj5byj")

Original thread:
fj5byj - Why I think we all need a second season of CLOY. (Spoilers)

Top 5 similar threads:
hsghrr - Second Season
olj6u1 - If there was a second season , what would you like to see?
lnyijx - Crash landing on you season 2 PLEASE!!
jz55ap - Just finished CLOY, I really hope they don't do second season and ruin the happy ending
ggdb8u - Season 2 Plot


## Comparing the Results

I compared the results for the same thread before and after removing stop words.

In [127]:
find_similar_threads("fj5byj")

Original thread:
fj5byj - Why I think we all need a second season of CLOY. (Spoilers)

Top 5 similar threads:
hsghrr - Second Season
g87z3m - Crash Landin on You: Potential season 2?
olj6u1 - If there was a second season , what would you like to see?
jg3333 - I think RJH and YSR get their happily ever after
jz55ap - Just finished CLOY, I really hope they don't do second season and ruin the happy ending


In [128]:
find_similar_threads_no_stopwords("fj5byj")

Original thread:
fj5byj - Why I think we all need a second season of CLOY. (Spoilers)

Top 5 similar threads:
hsghrr - Second Season
olj6u1 - If there was a second season , what would you like to see?
lnyijx - Crash landing on you season 2 PLEASE!!
jz55ap - Just finished CLOY, I really hope they don't do second season and ruin the happy ending
ggdb8u - Season 2 Plot


The results changed after I removed the stop words, but some of the same threads were still in the top five. "Second Season" stayed the most similar thread, while a few of the other results changed positions or were replaced by new threads.

I think stop words were affecting the results because removing them changed the ranking, but the results were still mostly related to the same topic. The comparison seems to focus more on meaningful words after removing common words like "the," "and," and "is."

## Reflection

The hardest part of this task was understanding the nested structure of the comments. At first I thought I could just loop through the comments, but then I saw that comments could have replies and those replies could have more replies.

Breaking the problem into smaller steps helped a lot. I first looked at one thread, then figured out how to collect the comments, then combined everything into text, and only after that moved on to the word vectors and cosine similarity.

I also thought the stop word comparison was interesting because it showed that the way I process the text can actually change which threads are considered the most similar.